# Wafer Map Defect Classification: Part 2, ResNet-18, ViT, Hybrid CNN-Transformer and Grad-CAM
**Before you run:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

When the upload button appears in the first cell, choose your **results.zip** from Part 1. It holds the exact train, validation and test splits and the baseline CNN scores, so every model is compared fairly.

This notebook:
1. Trains **ResNet-18** (deep CNN baseline) and a **Vision Transformer** (pure Transformer baseline)
2. Trains the **proposed Hybrid CNN-Transformer**: CNN layers capture local defect texture and Transformer layers capture the global layout across the wafer
3. Builds the final comparison table, with accuracy, macro F1, parameters, size and inference time
4. Produces **Grad-CAM** heatmaps showing where the hybrid model looks for each defect (the explainability part of your title)

Total run time is about 30–45 minutes on a T4.

In [ ]:
import os, time, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf, keras
from keras import layers as L
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from google.colab import files
up = files.upload()                        # choose results.zip from Part 1
zipfile.ZipFile(next(iter(up))).extractall('.')

SEED = 42; np.random.seed(SEED); tf.random.set_seed(SEED)
CLASSES = ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random', 'Scratch', 'none']
IMG, NC = 64, 9
os.makedirs('results2', exist_ok=True)

d = np.load('results/splits_64.npz')
onehot = lambda a: np.eye(3, dtype=np.uint8)[a]
X_tr, y_tr = onehot(d['X_tr']), d['y_tr']
X_va, y_va = onehot(d['X_va']), d['y_va']
X_te, y_te = onehot(d['X_te']), d['y_te']
print('train', X_tr.shape, 'val', X_va.shape, 'test', X_te.shape, '| GPU:', tf.config.list_physical_devices('GPU'))

## 1. Shared building blocks

In [ ]:
@keras.saving.register_keras_serializable()
class PositionEmbedding(L.Layer):
    """Learnable position embedding added to a sequence of tokens."""
    def build(self, input_shape):
        self.pos = self.add_weight(name='pos', shape=(1, input_shape[1], input_shape[2]), initializer='random_normal')
    def call(self, x):
        return x + self.pos

def transformer_block(x, dim, heads=4, mlp=256, drop=0.1, name=None):
    h = L.LayerNormalization(epsilon=1e-6)(x)
    h = L.MultiHeadAttention(num_heads=heads, key_dim=dim // heads, dropout=drop, name=name)(h, h)
    x = L.Add()([x, h])
    h = L.LayerNormalization(epsilon=1e-6)(x)
    h = L.Dense(mlp, activation='gelu')(h); h = L.Dropout(drop)(h)
    h = L.Dense(dim)(h); h = L.Dropout(drop)(h)
    return L.Add()([x, h])

def conv_bn(x, f, k=3, s=1, act=True, name=None):
    x = L.Conv2D(f, k, strides=s, padding='same', use_bias=False, name=name)(x)
    x = L.BatchNormalization()(x)
    return L.ReLU()(x) if act else x

## 2. Model A: ResNet-18 (adapted for 64×64 inputs)

In [ ]:
def basic_block(x, f, s):
    sc = x
    x = conv_bn(x, f, 3, s)
    x = conv_bn(x, f, 3, 1, act=False)
    if s != 1 or sc.shape[-1] != f:
        sc = conv_bn(sc, f, 1, s, act=False)
    return L.ReLU()(L.Add()([x, sc]))

def build_resnet18():
    inp = L.Input((IMG, IMG, 3))
    x = conv_bn(inp, 64, 3, 1)
    for f, s in [(64, 1), (128, 2), (256, 2), (512, 2)]:
        x = basic_block(x, f, s); x = basic_block(x, f, 1)
    x = L.GlobalAveragePooling2D()(x); x = L.Dropout(0.3)(x)
    return keras.Model(inp, L.Dense(NC, activation='softmax')(x), name='ResNet18')

## 3. Model B: Vision Transformer (ViT), trained from scratch
8×8 patches give 64 tokens of dimension 128, passed through 4 Transformer encoder blocks.

In [ ]:
def build_vit(dim=128, depth=4):
    inp = L.Input((IMG, IMG, 3))
    x = L.Conv2D(dim, 8, strides=8)(inp)          # patch embedding -> 8x8x128
    x = L.Reshape((-1, dim))(x)                    # 64 tokens
    x = PositionEmbedding()(x)
    for i in range(depth): x = transformer_block(x, dim)
    x = L.LayerNormalization(epsilon=1e-6)(x)
    x = L.GlobalAveragePooling1D()(x); x = L.Dropout(0.3)(x)
    return keras.Model(inp, L.Dense(NC, activation='softmax')(x), name='ViT')

## 4. Model C (proposed): Hybrid CNN-Transformer
- **CNN stem** (3 conv stages, 32→64→128 filters) learns local defect texture such as dots, arcs and clusters
- The 8×8×128 feature map becomes **64 tokens** with learnable position embeddings
- **2 Transformer encoder blocks** (4 heads) learn the global spatial relationships that tell Edge-Ring from Edge-Loc and Scratch from Loc
- The conv layer `last_conv` is the target layer for Grad-CAM

In [ ]:
def build_hybrid(dim=128, depth=2):
    inp = L.Input((IMG, IMG, 3))
    x = conv_bn(inp, 32); x = conv_bn(x, 32); x = L.MaxPooling2D()(x)          # 32x32
    x = conv_bn(x, 64); x = conv_bn(x, 64); x = L.MaxPooling2D()(x)            # 16x16
    x = conv_bn(x, dim); x = conv_bn(x, dim, name='last_conv'); x = L.MaxPooling2D()(x)  # 8x8
    x = L.Reshape((-1, dim))(x)
    x = PositionEmbedding()(x)
    for i in range(depth): x = transformer_block(x, dim, name=f'mhsa_{i}')
    x = L.LayerNormalization(epsilon=1e-6)(x)
    x = L.GlobalAveragePooling1D()(x); x = L.Dropout(0.3)(x)
    return keras.Model(inp, L.Dense(NC, activation='softmax')(x), name='Hybrid_CNN_Transformer')

## 5. Train and evaluate all three (same data, same settings)

In [ ]:
def run(model, epochs=40, lr=1e-3, bs=128):
    model.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
          keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)]
    t0 = time.time()
    h = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=epochs, batch_size=bs, callbacks=cb, verbose=2)
    tt = (time.time() - t0) / 60
    yp = model.predict(X_te, batch_size=256, verbose=0).argmax(1)
    fn = f'results2/{model.name}.keras'; model.save(fn)
    _ = model.predict(X_te[:64], batch_size=1, verbose=0)               # warm-up
    t0 = time.time(); _ = model.predict(X_te[:500], batch_size=1, verbose=0)
    ms = (time.time() - t0) / 500 * 1000
    row = {'model': model.name, 'params': int(model.count_params()), 'size_MB': round(os.path.getsize(fn) / 1e6, 2),
           'test_accuracy': round(accuracy_score(y_te, yp), 4), 'macro_F1': round(f1_score(y_te, yp, average='macro'), 4),
           'inference_ms_per_wafer': round(ms, 3), 'train_time_min': round(tt, 1)}
    pd.DataFrame(classification_report(y_te, yp, labels=list(range(NC)), target_names=CLASSES, digits=4, output_dict=True, zero_division=0)).T \
        .to_csv(f'results2/table_{model.name}_per_class.csv')
    print(row); return model, h, yp, row

results, preds, hists = [], {}, {}
for builder in [build_resnet18, build_vit, build_hybrid]:
    keras.backend.clear_session(); tf.random.set_seed(SEED)
    m, h, yp, row = run(builder())
    results.append(row); preds[m.name] = yp; hists[m.name] = h.history
hybrid = m   # the last model trained is the hybrid

## 6. Final comparison table (Table for the paper)

In [ ]:
base = pd.read_csv('results/table_model_comparison.csv')
table = pd.concat([base, pd.DataFrame(results)], ignore_index=True)
table.to_csv('results2/table_final_comparison.csv', index=False)
table

In [ ]:
# Per-class F1 for every model (shows where the hybrid helps: Scratch, Loc, Donut)
f1s = {'Baseline CNN': pd.read_csv('results/table_cnn_per_class.csv', index_col=0).loc[CLASSES, 'f1-score'].values}
for name, yp in preds.items(): f1s[name] = f1_score(y_te, yp, average=None, labels=range(NC))
f1df = pd.DataFrame(f1s, index=CLASSES)
f1df.round(4).to_csv('results2/table_per_class_f1_all_models.csv')
ax = f1df.plot.bar(figsize=(12, 4.2), width=0.8)
ax.set_ylim(0.5, 1.0); ax.set_ylabel('F1-score'); ax.set_title('Per-class F1-score by model')
plt.xticks(rotation=0); plt.legend(loc='lower right', ncol=4); plt.tight_layout()
plt.savefig('results2/fig_per_class_f1.png', dpi=300); plt.show()
f1df.round(3)

In [ ]:
# Confusion matrix of the proposed model
cm = confusion_matrix(y_te, preds['Hybrid_CNN_Transformer'], normalize='true')
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Proposed Hybrid CNN-Transformer: normalised confusion matrix')
plt.tight_layout(); plt.savefig('results2/fig_hybrid_confusion_matrix.png', dpi=300); plt.show()

# Validation accuracy curves for all models
plt.figure(figsize=(7, 4))
for n, hh in hists.items(): plt.plot(hh['val_accuracy'], label=n)
plt.xlabel('Epoch'); plt.ylabel('Validation accuracy'); plt.legend(); plt.title('Validation accuracy by model')
plt.tight_layout(); plt.savefig('results2/fig_val_accuracy_all.png', dpi=300); plt.show()

## 7. Explainability with Grad-CAM
Grad-CAM uses the gradients flowing into `last_conv` to highlight which wafer regions drove each prediction. A trustworthy model should light up the ring for Edge-Ring, the line for Scratch and the middle for Center.

In [ ]:
grad_model = keras.Model(hybrid.inputs, [hybrid.get_layer('last_conv').output, hybrid.output])

def gradcam(img, cls=None):
    x = tf.convert_to_tensor(img[None].astype('float32'))
    with tf.GradientTape() as tape:
        conv, pred = grad_model(x)
        if cls is None: cls = int(tf.argmax(pred[0]))
        score = pred[:, cls]
    g = tape.gradient(score, conv)
    w = tf.reduce_mean(g, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, axis=-1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return tf.image.resize(cam[..., None], (IMG, IMG)).numpy()[..., 0], cls, float(pred[0, cls])

yp = preds['Hybrid_CNN_Transformer']
fig, ax = plt.subplots(2, 9, figsize=(20, 5))
for c, name in enumerate(CLASSES):
    ok = np.where((y_te == c) & (yp == c))[0]
    i = ok[0] if len(ok) else np.where(y_te == c)[0][0]
    cam, pc, p = gradcam(X_te[i])
    wafer = X_te[i].argmax(-1)
    ax[0, c].imshow(wafer, cmap='viridis'); ax[0, c].set_title(name); ax[0, c].axis('off')
    ax[1, c].imshow(wafer, cmap='gray'); ax[1, c].imshow(cam, cmap='jet', alpha=0.5)
    ax[1, c].set_title(f'{CLASSES[pc]} ({p:.2f})', fontsize=9); ax[1, c].axis('off')
ax[0, 0].text(-20, 32, 'Wafer', rotation=90, va='center'); ax[1, 0].text(-20, 32, 'Grad-CAM', rotation=90, va='center')
plt.tight_layout(); plt.savefig('results2/fig_gradcam_hybrid.png', dpi=300); plt.show()

In [ ]:
# Grad-CAM on misclassified wafers (good for the Discussion section: shows *why* the model failed)
wrong = np.where(yp != y_te)[0][:6]
if len(wrong):
    fig, ax = plt.subplots(1, len(wrong), figsize=(3 * len(wrong), 3.2))
    ax = np.atleast_1d(ax)
    for a, i in zip(ax, wrong):
        cam, pc, p = gradcam(X_te[i])
        a.imshow(X_te[i].argmax(-1), cmap='gray'); a.imshow(cam, cmap='jet', alpha=0.5)
        a.set_title(f'True {CLASSES[y_te[i]]}\nPred {CLASSES[pc]} ({p:.2f})', fontsize=9); a.axis('off')
    plt.tight_layout(); plt.savefig('results2/fig_gradcam_errors.png', dpi=300); plt.show()

## 8. Download everything

In [ ]:
import shutil
shutil.make_archive('results2', 'zip', 'results2')
files.download('results2.zip')